# 🏦 Optimizador de Patrimonio en USD — v2.0
### Markowitz · Riesgo vs. Retorno · Proyección hacia la Meta · Datos Bull Market Abril 2026

> **Instrucción de uso:** Editar únicamente la celda `PARÁMETROS GLOBALES` y ejecutar todas las celdas en orden (`Kernel > Restart & Run All`).

---
**Autor:** Fran | **Broker:** Bull Market ALyC | **Capital líquido:** $1.782 USDC

#### Mejoras v2.0
- ✅ **Datos reales de fondos Bull Market** (Abril 2026) — ONs corporativas HD, FCIs USD, Acciones
- ✅ **Aporte mensual** como parámetro en la proyección
- ✅ **Análisis de sensibilidad** — cómo varía la fecha de meta según APR
- ✅ **FCIs Corporativos HD** como clase de activo (Allaria Dólar Dinámico, BM Ahorro Dólares, etc.)
- ✅ **Scraping robusto** con fallback a datos del informe Bull Market
- ✅ **Radar de riesgo** por instrumento
- ✅ **Disclaimer mejorado** con nota legal

## ⚙️ 0. Instalación de dependencias

In [ ]:
# Ejecutar sólo la primera vez (o en entornos nuevos)
# !pip install yfinance pyportfolioopt requests beautifulsoup4 plotly pandas numpy scipy --quiet
print("✅ Dependencias listas")

## 📦 1. Imports

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pypfopt import EfficientFrontier, risk_models, expected_returns
import warnings
warnings.filterwarnings("ignore")
from IPython.display import display, HTML
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from datetime import datetime

print(f"✅ Imports completados — {datetime.now().strftime('%Y-%m-%d %H:%M')}")

## 🎛️ 2. PARÁMETROS GLOBALES
> **⚠️ Única celda a editar.** Cambiá cualquier valor y re-ejecutá todo el notebook.

In [ ]:
# ════════════════════════════════════════════════════════
#  CAPITAL Y META
# ════════════════════════════════════════════════════════
CAPITAL_LIQUIDO_USD   = 1_782    # USDC en Binance actualmente
OTROS_ACTIVOS_USD     = 663      # Soja + bonos (2445 - 1782)
META_USD              = 3_000    # Objetivo de patrimonio neto total
HORIZONTE_MESES       = 36       # Meses máximos a proyectar
APORTE_MENSUAL_USD    = 55       # Tu aporte real actual
APORTES_ESCENARIOS    = [55, 100, 150]  # Hoy / pasantía / máximo

# ════════════════════════════════════════════════════════
#  COMISIONES BULL MARKET (editables)
# ════════════════════════════════════════════════════════
COMISION_BROKER_ON        = 0.005   # 0.50%  comisión broker en ONs
SPREAD_ON                 = 0.0075  # 0.75%  spread bid/ask ONs
COMISION_CEDEAR_ENTRADA   = 0.005   # 0.50%  entrada CEDEARs
COMISION_CEDEAR_SALIDA    = 0.005   # 0.50%  salida CEDEARs
TER_FCI_ANUAL             = 0.010   # ~1.00% TER anual estimado FCIs USD

# ════════════════════════════════════════════════════════
#  PARÁMETROS DEL MODELO
# ════════════════════════════════════════════════════════
INFLACION_USD_ANUAL       = 0.030   # 3.0%  inflación USD (retorno real)
PERIODO_HISTORICO_CEDEAR  = "3y"    # historial para SPY / QQQ
TICKERS_CEDEAR            = ["SPY", "QQQ"]
RISK_FREE_RATE            = 0.045   # 4.5%  T-Bills USA

# ════════════════════════════════════════════════════════
#  DATOS BULL MARKET ABRIL 2026 — Fondos Corporativos HD
#  Fuente: Informe de Fondos Abril 2026, Research BMB
#  Retornos en USD (medidos en dólares), hasta 13/04/2026
# ════════════════════════════════════════════════════════
FONDOS_CORP_HD = {
    "Compass Best Ideas":       {"ret_12m": 0.1101, "ret_6m": 0.0621, "ret_3m": 0.0259, "vol": 0.038},
    "BM Ahorro Dólares":        {"ret_12m": 0.0650, "ret_6m": 0.0475, "ret_3m": 0.0253, "vol": 0.021},
    "Allaria Dólar Dinámico":   {"ret_12m": 0.1100, "ret_6m": 0.0648, "ret_3m": 0.0210, "vol": 0.040},
    "MegaQM Latam Corporativo": {"ret_12m": 0.0487, "ret_6m": 0.0109, "ret_3m": 0.0042, "vol": 0.025},
}

# ════════════════════════════════════════════════════════
#  APRs DE REFERENCIA (fallback si falla una API)
# ════════════════════════════════════════════════════════
APR_STAKING_BINANCE       = 0.0114  # 1.14%  Binance USDC (situación actual)
APR_STAKING_ALTERNATIVO   = 0.045   # 4.50%  CeFi alternativo (ej. Coinbase)
APR_ON_FALLBACK           = 0.080   # 8.00%  TIR promedio ONs USD
APR_FCI_FALLBACK          = 0.065   # 6.50%  FCIs Corp HD USD promedio (actualizado BM Abril)

# ════════════════════════════════════════════════════════
#  PESOS DEL RISK SCORE (deben sumar 1.0)
# ════════════════════════════════════════════════════════
PESOS_RISK = {
    "contraparte":  0.30,   # riesgo de default / custodia
    "volatilidad":  0.25,   # volatilidad histórica de precio
    "liquidez":     0.20,   # facilidad/días para salir
    "regulatorio":  0.15,   # riesgo Argentina + regulatorio
    "lockup":       0.10,   # inmovilización de capital
}

# ════════════════════════════════════════════════════════
#  RESTRICCIONES DE PESOS EN EL PORTAFOLIO
# ════════════════════════════════════════════════════════
PESO_MINIMO_POR_ACTIVO  = 0.05   # mínimo 5% por instrumento
PESO_MAXIMO_POR_ACTIVO  = 0.55   # máximo 55% por instrumento

# ─── Verificación ────────────────────────────────────────
patrimonio_actual = CAPITAL_LIQUIDO_USD + OTROS_ACTIVOS_USD
brecha            = META_USD - patrimonio_actual

print("═" * 58)
print("  📋 RESUMEN DE SITUACIÓN INICIAL")
print("═" * 58)
print(f"  Capital líquido (USDC)    : ${CAPITAL_LIQUIDO_USD:>8,.2f}")
print(f"  Otros activos (soja/bonos): ${OTROS_ACTIVOS_USD:>8,.2f}")
print(f"  Patrimonio total actual   : ${patrimonio_actual:>8,.2f}")
print(f"  Meta de patrimonio        : ${META_USD:>8,.2f}")
print(f"  Brecha a cubrir           : ${brecha:>8,.2f}  ({brecha/patrimonio_actual:.1%})")
print(f"  Horizonte máximo          : {HORIZONTE_MESES} meses")
print(f"  Aporte mensual (base)     : ${APORTE_MENSUAL_USD:>8,.2f}")
print("═" * 58)
print(f"  Escenarios de aporte     : {APORTES_ESCENARIOS}")
assert abs(sum(PESOS_RISK.values()) - 1.0) < 1e-9, "ERROR: pesos del Risk Score deben sumar 1.0"
print("  ✅ Parámetros validados correctamente")

## 📡 3. Datos de Mercado

### 3A. CEDEARs — SPY / QQQ via `yfinance`
> Usamos los ETFs originales en USD (no el precio en pesos del CEDEAR) para evitar distorsiones cambiarias.

In [ ]:
print("Descargando histórico de ETFs (esto puede tardar unos segundos)...")

prices_raw = yf.download(TICKERS_CEDEAR, period=PERIODO_HISTORICO_CEDEAR,
                          auto_adjust=True, progress=False)

# Manejar tanto MultiIndex como columna simple
if isinstance(prices_raw.columns, pd.MultiIndex):
    prices_df = prices_raw["Close"]
else:
    prices_df = prices_raw[["Close"]].rename(columns={"Close": TICKERS_CEDEAR[0]})

prices_df.dropna(inplace=True)
daily_ret = prices_df.pct_change().dropna()

annual_ret_cedear = daily_ret.mean()  * 252
annual_vol_cedear = daily_ret.std()   * np.sqrt(252)
comision_cedear_total = COMISION_CEDEAR_ENTRADA + COMISION_CEDEAR_SALIDA

print(f"\n  Período: {prices_df.index[0].date()} → {prices_df.index[-1].date()}  ({len(prices_df)} días)")
print(f"\n  {'Ticker':<12} {'Retorno Bruto':>15} {'Comisiones':>12} {'Retorno Neto':>14} {'Volatilidad':>12}")
print("  " + "-"*67)
for t in TICKERS_CEDEAR:
    if t in annual_ret_cedear:
        rb   = annual_ret_cedear[t]
        neto = rb - comision_cedear_total - INFLACION_USD_ANUAL
        vol  = annual_vol_cedear[t]
        print(f"  {t:<12} {rb:>14.2%} {comision_cedear_total:>11.2%} {neto:>13.2%} {vol:>11.2%}")

print("\n✅ Datos de CEDEARs cargados")

### 3B. FCIs Corporativos en USD — Datos Bull Market Abril 2026

> **🆕 Cambio v2:** En lugar de depender de la API de CAFCI (que requiere auth y falla con 401),
> usamos los **retornos reales** del Informe de Fondos de Bull Market Research (Abril 2026).
> Los fondos corporativos hard-dollar son los más relevantes para tu perfil.
>
> Se mantiene el intento de API como enriquecimiento, pero el **fallback es data real** del research.

In [ ]:
def fetch_fcis_usd():
    """
    Intenta obtener FCIs desde CAFCI. Si falla, usa datos reales
    del Informe Bull Market Abril 2026 (Fondos Corporativos HD).
    """
    base = "https://api.cafci.org.ar"
    headers = {
        "User-Agent"      : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept"          : "application/json, text/plain, */*",
        "Origin"          : "https://www.cafci.org.ar",
        "Referer"         : "https://www.cafci.org.ar/",
    }

    intentos = [
        f"{base}/fondo?estado=1&limit=30",
        f"{base}/fondo?limit=30",
        f"{base}/fondo",
    ]

    for url in intentos:
        try:
            resp = requests.get(url, headers=headers, timeout=10, verify=False)
            resp.raise_for_status()
            raw = resp.json()
            items = raw if isinstance(raw, list) else raw.get("data", [])

            fondos = []
            for f in items:
                moneda = ""
                if isinstance(f.get("moneda"), dict):
                    moneda = f["moneda"].get("nombre", "")
                elif isinstance(f.get("moneda"), str):
                    moneda = f["moneda"]
                if moneda and "USD" not in moneda.upper() and "DOLAR" not in moneda.upper():
                    continue
                fondos.append({
                    "nombre": str(f.get("nombre", "—"))[:50],
                    "tipo"  : f.get("tipo_fondo", {}).get("nombre", "—") if isinstance(f.get("tipo_fondo"), dict) else "—",
                    "moneda": moneda or "USD",
                })

            if fondos:
                df = pd.DataFrame(fondos)
                df["apr_bruto"]   = APR_FCI_FALLBACK
                df["apr_neto"]    = APR_FCI_FALLBACK - TER_FCI_ANUAL - INFLACION_USD_ANUAL
                df["volatilidad"] = 0.030
                print(f"  ✅ CAFCI OK [{resp.status_code}] — {len(df)} fondos USD")
                return df
        except Exception as e:
            print(f"  ↳ CAFCI intento fallido: {type(e).__name__}")
            continue

    # ── Fallback: Datos REALES del Informe Bull Market Abril 2026 ──
    print("  ⚠️  CAFCI no disponible — usando datos reales Bull Market Research Abril 2026")
    fondos_bm = []
    for nombre, datos in FONDOS_CORP_HD.items():
        fondos_bm.append({
            "nombre"      : nombre,
            "tipo"        : "Renta Fija Corp HD",
            "moneda"      : "USD",
            "apr_bruto"   : datos["ret_12m"],
            "apr_neto"    : datos["ret_12m"] - TER_FCI_ANUAL - INFLACION_USD_ANUAL,
            "ret_6m"      : datos["ret_6m"],
            "ret_3m"      : datos["ret_3m"],
            "volatilidad" : datos["vol"],
        })
    return pd.DataFrame(fondos_bm)

df_fcis = fetch_fcis_usd()
cols_show = [c for c in ["nombre", "tipo", "apr_bruto", "apr_neto", "ret_6m", "ret_3m"] if c in df_fcis.columns]
fmt = {c: "{:.2%}" for c in cols_show if c not in ["nombre", "tipo"]}
display(df_fcis[cols_show].head(8).style.format(fmt).set_caption("FCIs USD — Fondos Corporativos HD"))

### 3C. Obligaciones Negociables — BYMA Open Data

> **🆕 Cambio v2:** El endpoint de BYMA requiere un body POST específico que no está documentado
> públicamente. Se agrega el intento con POST + body vacío / con filtros, y si falla,
> se usa data del informe Bull Market (los fondos corporativos HD tienen ONs como
> principales tenencias: Pampa Energía, John Deere, Loma Negra, Otaméricas, YPF, etc.).

In [ ]:
def fetch_ons_byma(max_ons=25):
    """
    Obtiene ONs en USD desde BYMA Open Data.
    Intenta GET y POST; fallback a ONs de referencia del informe BM.
    """
    headers = {
        "User-Agent"     : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Accept"         : "application/json, text/plain, */*",
        "Accept-Language": "es-AR,es;q=0.9",
        "Referer"        : "https://open.bymadata.com.ar/",
        "Origin"         : "https://open.bymadata.com.ar",
        "Content-Type"   : "application/json",
    }

    # ── Intento 1: BYMA POST con body (el endpoint requiere POST) ──
    byma_post_url = "https://open.bymadata.com.ar/vanoms-be-core/rest/api/bymadata/free/ons"
    post_bodies = [
        {"Content-Type": "application/json"},  # empty-ish
        {"excludeZeroPx498": True, "T2": True},
        {},
    ]

    for body in post_bodies:
        try:
            resp = requests.post(byma_post_url, json=body, headers=headers, timeout=15, verify=False)
            if resp.status_code == 200:
                data = resp.json()
                items = data if isinstance(data, list) else data.get("data", data.get("content", []))
                ons = []
                for item in items[:max_ons]:
                    moneda = str(item.get("denominationCcy", item.get("currency", ""))).upper()
                    tir = item.get("tir") or item.get("yield") or item.get("yieldRate")
                    if moneda in ("USD", "D", "U", "DOLAR") and tir:
                        try:
                            ons.append({
                                "ticker"     : item.get("symbol", item.get("ticker", "—")),
                                "descripcion": str(item.get("description", item.get("name", "—")))[:42],
                                "tir_bruta"  : float(tir) / 100 if float(tir) > 1 else float(tir),
                                "vencimiento": item.get("maturityDate", "—"),
                                "precio"     : float(item.get("settlementPrice", item.get("price", 0)) or 0),
                            })
                        except (ValueError, TypeError):
                            pass
                if ons:
                    df = pd.DataFrame(ons)
                    df["tir_neta"]    = df["tir_bruta"] - COMISION_BROKER_ON - SPREAD_ON - INFLACION_USD_ANUAL
                    df["volatilidad"] = 0.020
                    print(f"  ✅ BYMA OK [{resp.status_code}] — {len(df)} ONs en USD")
                    return df
            print(f"  ↳ BYMA POST [{resp.status_code}] sin ONs USD útiles")
        except Exception as e:
            print(f"  ↳ BYMA POST ERR: {type(e).__name__}")

    # ── Intento 2: GET endpoints alternativos ──
    byma_get_urls = [
        "https://open.bymadata.com.ar/vanoms-be-core/rest/api/bymadata/free/obligaciones-negociables",
        "https://open.bymadata.com.ar/vanoms-be-core/rest/api/bymadata/free/ons-negociadas-hoy",
    ]
    for url in byma_get_urls:
        try:
            resp = requests.get(url, headers=headers, timeout=15, verify=False)
            if resp.status_code == 200:
                data = resp.json()
                items = data if isinstance(data, list) else data.get("data", [])
                ons = []
                for item in items[:max_ons]:
                    moneda = str(item.get("currency", "")).upper()
                    tir    = item.get("yield") or item.get("tir")
                    if moneda in ("USD", "D", "U") and tir:
                        try:
                            ons.append({
                                "ticker"     : item.get("symbol", "—"),
                                "descripcion": str(item.get("description", "—"))[:42],
                                "tir_bruta"  : float(tir) / 100 if float(tir) > 1 else float(tir),
                                "vencimiento": item.get("maturityDate", "—"),
                                "precio"     : float(item.get("price", 0) or 0),
                            })
                        except (ValueError, TypeError):
                            pass
                if ons:
                    df = pd.DataFrame(ons)
                    df["tir_neta"]    = df["tir_bruta"] - COMISION_BROKER_ON - SPREAD_ON - INFLACION_USD_ANUAL
                    df["volatilidad"] = 0.020
                    print(f"  ✅ BYMA GET OK — {len(df)} ONs en USD")
                    return df
        except Exception as e:
            print(f"  ↳ BYMA GET ERR: {type(e).__name__}")

    # ── Fallback: ONs de referencia del Informe Bull Market ──
    # Principales tenencias de los Fondos Corporativos HD (pág. 12 del informe)
    print("  ⚠️  BYMA no disponible — usando ONs de referencia (tenencias fondos BM Abril 2026)")
    ons_ref = [
        {"ticker": "MGCSD",  "descripcion": "Pampa Energía Cl.27",     "tir_bruta": 0.078, "vencimiento": "2027", "precio": 101.2},
        {"ticker": "HJCKD",  "descripcion": "John Deere Credit Cl.19", "tir_bruta": 0.065, "vencimiento": "2026", "precio": 100.5},
        {"ticker": "LOC6D",  "descripcion": "Loma Negra Cl.6",         "tir_bruta": 0.072, "vencimiento": "2027", "precio": 99.8},
        {"ticker": "OTS6D",  "descripcion": "Otaméricas Ebytem Cl.VI", "tir_bruta": 0.085, "vencimiento": "2028", "precio": 98.5},
        {"ticker": "YFCDO",  "descripcion": "YPF Luz Cl.12",           "tir_bruta": 0.082, "vencimiento": "2027", "precio": 100.1},
        {"ticker": "TLC6D",  "descripcion": "Telecom Argentina Cl.15", "tir_bruta": 0.070, "vencimiento": "2027", "precio": 101.0},
        {"ticker": "VSCHO",  "descripcion": "Vista Oil & Gas Cl.16",   "tir_bruta": 0.075, "vencimiento": "2028", "precio": 99.2},
    ]
    df = pd.DataFrame(ons_ref)
    df["tir_neta"]    = df["tir_bruta"] - COMISION_BROKER_ON - SPREAD_ON - INFLACION_USD_ANUAL
    df["volatilidad"] = 0.020
    return df

df_ons = fetch_ons_byma()
display(df_ons[["ticker", "descripcion", "tir_bruta", "tir_neta", "vencimiento"]]
        .head(8).style
        .format({"tir_bruta": "{:.2%}", "tir_neta": "{:.2%}"})
        .set_caption("ONs USD — BYMA / Bull Market Research"))

### 3D. Staking / Earn CeFi — Referencia de Mercado

In [ ]:
staking_data = [
    {"plataforma": "Binance (actual)",      "activo": "USDC", "apr_bruto": APR_STAKING_BINANCE,     "tipo": "Flexible"},
    {"plataforma": "Binance Flexible",      "activo": "USDT", "apr_bruto": 0.030,                   "tipo": "Flexible"},
    {"plataforma": "Coinbase",              "activo": "USDC", "apr_bruto": APR_STAKING_ALTERNATIVO,  "tipo": "Flexible"},
    {"plataforma": "Aave / Arbitrum",       "activo": "USDC", "apr_bruto": 0.060,                   "tipo": "DeFi / Flexible"},
    {"plataforma": "Ondo Finance",          "activo": "USDY", "apr_bruto": 0.052,                   "tipo": "Flexible"},
]

df_staking = pd.DataFrame(staking_data)
df_staking["apr_neto"] = df_staking["apr_bruto"] - INFLACION_USD_ANUAL
df_staking["volatilidad"] = [0.01, 0.01, 0.05, 0.08, 0.04]

display(df_staking.style
        .format({"apr_bruto": "{:.2%}", "apr_neto": "{:.2%}"})
        .set_caption("Opciones de Staking / Earn"))
print("\n✅ Datos de staking listos")

## ⚖️ 4. Risk Score Personalizado

Cada dimensión se puntúa de **0 (mínimo riesgo) a 10 (máximo riesgo)**.
El score final es un promedio ponderado según `PESOS_RISK`.

| Dimensión | Descripción |
|---|---|
| **Contraparte** | Riesgo de default del emisor / custodia de activos |
| **Volatilidad** | Oscilación histórica del precio |
| **Liquidez** | Facilidad y velocidad para salir de la posición |
| **Regulatorio** | Exposición al marco regulatorio argentino |
| **Lock-up** | Días/meses de inmovilización de capital |

> **🆕 v2:** Se agrega `FCI Corp HD` como clase separada de `FCI USD` genérico.

In [ ]:
# ─── Scores manuales por dimensión (0 = menor riesgo, 10 = mayor riesgo) ────
#              activo       contra  volatil  liquidez  regulat  lockup
risk_raw = {
    "ON USD"        : [2,       1,       4,        3,       5],
    "SPY CEDEAR"    : [1,       7,       2,        3,       1],
    "QQQ CEDEAR"    : [1,       8,       2,        3,       1],
    "FCI Corp HD"   : [2,       2,       2,        2,       3],
    "Staking CeFi"  : [7,       1,       3,        4,       2],
}
dims = list(PESOS_RISK.keys())

df_risk = pd.DataFrame(risk_raw, index=dims).T.reset_index()
df_risk.columns = ["activo"] + dims

for dim, peso in PESOS_RISK.items():
    df_risk[f"w_{dim}"] = df_risk[dim] * peso

df_risk["risk_score"] = df_risk[[f"w_{d}" for d in dims]].sum(axis=1)
df_risk = df_risk[["activo", "risk_score"] + dims].sort_values("risk_score")

print("📊 Risk Scores calculados:")
display(
    df_risk.style
    .format({**{d: "{:.0f}" for d in dims}, "risk_score": "{:.2f}"})
    .background_gradient(subset="risk_score", cmap="RdYlGn_r")
    .set_caption("Risk Score por instrumento (menor = más seguro)")
)

## 📋 5. Tabla Unificada de Activos

In [ ]:
# ─── Retornos netos por instrumento ─────────────────────────────────────────
best_on_neto = df_ons["tir_neta"].max() if "tir_neta" in df_ons.columns else (
    APR_ON_FALLBACK - COMISION_BROKER_ON - SPREAD_ON - INFLACION_USD_ANUAL)
best_on_bruto = df_ons["tir_bruta"].max() if "tir_bruta" in df_ons.columns else APR_ON_FALLBACK

# FCI Corp HD: usamos el promedio ponderado de los fondos del informe BM
fci_hd_bruto = np.mean([d["ret_12m"] for d in FONDOS_CORP_HD.values()])
fci_hd_vol   = np.mean([d["vol"]     for d in FONDOS_CORP_HD.values()])
fci_hd_neto  = fci_hd_bruto - TER_FCI_ANUAL - INFLACION_USD_ANUAL

stk_neto  = APR_STAKING_ALTERNATIVO - INFLACION_USD_ANUAL

# Retornos CEDEARs (desde yfinance)
spy_bruto = float(annual_ret_cedear.get("SPY", 0))
qqq_bruto = float(annual_ret_cedear.get("QQQ", 0))
spy_neto  = spy_bruto - comision_cedear_total - INFLACION_USD_ANUAL
qqq_neto  = qqq_bruto - comision_cedear_total - INFLACION_USD_ANUAL
spy_vol   = float(annual_vol_cedear.get("SPY", 0.15))
qqq_vol   = float(annual_vol_cedear.get("QQQ", 0.18))

assets_info = [
    {"activo": "ON USD",        "retorno_bruto": best_on_bruto,           "retorno_neto": best_on_neto,  "volatilidad": 0.020},
    {"activo": "SPY CEDEAR",    "retorno_bruto": spy_bruto,               "retorno_neto": spy_neto,      "volatilidad": spy_vol},
    {"activo": "QQQ CEDEAR",    "retorno_bruto": qqq_bruto,               "retorno_neto": qqq_neto,      "volatilidad": qqq_vol},
    {"activo": "FCI Corp HD",   "retorno_bruto": fci_hd_bruto,            "retorno_neto": fci_hd_neto,   "volatilidad": fci_hd_vol},
    {"activo": "Staking CeFi",  "retorno_bruto": APR_STAKING_ALTERNATIVO, "retorno_neto": stk_neto,      "volatilidad": 0.050},
]

df_assets = pd.DataFrame(assets_info)
df_assets = df_assets.merge(df_risk[["activo", "risk_score"]], on="activo", how="left")
df_assets["sharpe"] = (
    (df_assets["retorno_neto"] - RISK_FREE_RATE) / df_assets["volatilidad"]
).clip(lower=-5)
df_assets = df_assets.sort_values("retorno_neto", ascending=False).reset_index(drop=True)

print("📋 Tabla unificada de activos (con datos BM Abril 2026):")
display(
    df_assets.style
    .format({
        "retorno_bruto": "{:.2%}", "retorno_neto": "{:.2%}",
        "volatilidad"  : "{:.2%}", "risk_score"  : "{:.2f}",
        "sharpe"       : "{:.2f}",
    })
    .background_gradient(subset="retorno_neto", cmap="RdYlGn")
    .background_gradient(subset="risk_score", cmap="RdYlGn_r")
    .set_caption("Retorno Neto Real · Volatilidad · Risk Score · Sharpe")
)

## 📈 6. Optimización de Markowitz

### Metodología
Para activos con precios históricos (SPY, QQQ) usamos la serie real de `yfinance`.
Para instrumentos de renta fija / staking generamos series sintéticas con el retorno
y la volatilidad estimados, con correlación **cero** respecto a renta variable
(supuesto conservador).

> **🆕 v2:** Se agrega `FCI_Corp_HD` con datos reales de volatilidad del informe BM.

In [ ]:
np.random.seed(42)
n_dias = len(daily_ret)

# ─── Series sintéticas para renta fija y staking ─────────────────────────────
def daily_normal(annual_return, annual_vol, n):
    """Serie de retornos diarios normales con media y vol anualizados."""
    mu_d    = (1 + annual_return) ** (1/252) - 1
    sigma_d = annual_vol / np.sqrt(252)
    return np.random.normal(mu_d, sigma_d, n)

synth_on      = daily_normal(best_on_bruto,            0.020,      n_dias)
synth_fci_hd  = daily_normal(fci_hd_bruto,             fci_hd_vol, n_dias)
synth_stk     = daily_normal(APR_STAKING_ALTERNATIVO,  0.050,      n_dias)

# ─── DataFrame combinado de retornos diarios ─────────────────────────────────
all_ret = daily_ret.copy()
all_ret.columns = [c if isinstance(c, str) else c[0] for c in all_ret.columns]
all_ret["ON_USD"]        = synth_on
all_ret["FCI_Corp_HD"]   = synth_fci_hd
all_ret["Staking_CeFi"]  = synth_stk

# Normalizar nombres de columnas
col_map = {}
for c in all_ret.columns:
    if "SPY" in str(c): col_map[c] = "SPY_CEDEAR"
    elif "QQQ" in str(c): col_map[c] = "QQQ_CEDEAR"
    else: col_map[c] = c
all_ret.rename(columns=col_map, inplace=True)

# ─── Expected returns y covarianza ───────────────────────────────────────────
mu_pf = expected_returns.mean_historical_return(all_ret, returns_data=True, frequency=252)
S_pf  = risk_models.sample_cov(all_ret, returns_data=True, frequency=252)

# Override con valores brutos reales
mu_pf["ON_USD"]        = best_on_bruto
mu_pf["FCI_Corp_HD"]   = fci_hd_bruto
mu_pf["Staking_CeFi"]  = APR_STAKING_ALTERNATIVO

print("Expected Returns (anualizados):")
for k, v in mu_pf.items():
    print(f"   {k:<18}: {v:.2%}")

# ─── Frontera eficiente: Máximo Sharpe ───────────────────────────────────────
ef_ms = EfficientFrontier(mu_pf, S_pf)
ef_ms.add_constraint(lambda w: w >= PESO_MINIMO_POR_ACTIVO)
ef_ms.add_constraint(lambda w: w <= PESO_MAXIMO_POR_ACTIVO)
ef_ms.max_sharpe(risk_free_rate=RISK_FREE_RATE)
w_sharpe = ef_ms.clean_weights()
ret_ms, vol_ms, sharpe_ms = ef_ms.portfolio_performance(risk_free_rate=RISK_FREE_RATE)

# ─── Frontera eficiente: Mínima Volatilidad ──────────────────────────────────
ef_mv = EfficientFrontier(mu_pf, S_pf)
ef_mv.add_constraint(lambda w: w >= PESO_MINIMO_POR_ACTIVO)
ef_mv.add_constraint(lambda w: w <= PESO_MAXIMO_POR_ACTIVO)
ef_mv.min_volatility()
w_min_vol = ef_mv.clean_weights()
ret_mv, vol_mv, sharpe_mv = ef_mv.portfolio_performance(risk_free_rate=RISK_FREE_RATE)

# ─── Resumen ─────────────────────────────────────────────────────────────────
print("\n" + "═"*58)
print("  PORTAFOLIO MÁXIMO SHARPE")
print("═"*58)
for k, v in w_sharpe.items():
    print(f"  {k:<18}: {v:.1%}  = ${CAPITAL_LIQUIDO_USD * v:>8,.0f} USD")
print(f"  {'Retorno esperado':<20}: {ret_ms:.2%}")
print(f"  {'Volatilidad':<20}: {vol_ms:.2%}")
print(f"  {'Sharpe ratio':<20}: {sharpe_ms:.2f}")

print("\n" + "═"*58)
print("  PORTAFOLIO MÍNIMA VOLATILIDAD")
print("═"*58)
for k, v in w_min_vol.items():
    print(f"  {k:<18}: {v:.1%}  = ${CAPITAL_LIQUIDO_USD * v:>8,.0f} USD")
print(f"  {'Retorno esperado':<20}: {ret_mv:.2%}")
print(f"  {'Volatilidad':<20}: {vol_mv:.2%}")
print(f"  {'Sharpe ratio':<20}: {sharpe_mv:.2f}")

---
# 🎯 TU PLAN — Resumen Personal
> Lo que importa: dónde estás, a dónde vas, y qué hacer hoy.
> El detalle técnico está más abajo si querés profundizar.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  DASHBOARD PRINCIPAL — Lo primero que ves al abrir
# ═══════════════════════════════════════════════════════════════

# Tus datos
pat_actual = CAPITAL_LIQUIDO_USD + OTROS_ACTIVOS_USD
pct_avance = pat_actual / META_USD * 100
brecha = META_USD - pat_actual

# Mejor escenario realista: Max Sharpe + tu aporte actual
ret_plan = ret_ms - INFLACION_USD_ANUAL  # retorno neto real del portafolio óptimo

def proyectar(capital_liq, ret_anual, meses, aporte_mensual=0, otros=OTROS_ACTIVOS_USD):
    ret_mensual = (1 + ret_anual) ** (1/12) - 1
    serie = [capital_liq + otros]
    capital = capital_liq
    for m in range(1, meses + 1):
        capital = capital * (1 + ret_mensual) + aporte_mensual
        serie.append(capital + otros)
    return serie

def mes_meta(serie, meta):
    for i, v in enumerate(serie):
        if v >= meta: return i
    return None

serie_plan = proyectar(CAPITAL_LIQUIDO_USD, ret_plan, 60, aporte_mensual=APORTE_MENSUAL_USD)
m_plan = mes_meta(serie_plan, META_USD)

# ── Gauge de progreso ──
fig_dash = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "indicator"}, {"type": "indicator"}]],
    subplot_titles=["Avance hacia la meta", "Meses estimados"],
)

fig_dash.add_trace(go.Indicator(
    mode="gauge+number",
    value=pct_avance,
    number=dict(suffix="%", font=dict(size=36, color="white")),
    gauge=dict(
        axis=dict(range=[0, 100], tickwidth=1, tickcolor="#555"),
        bar=dict(color="#00d4aa", thickness=0.75),
        bgcolor="#1a1a2e",
        borderwidth=1, bordercolor="#333",
        steps=[
            dict(range=[0, 33], color="#2e1a1a"),
            dict(range=[33, 66], color="#2e2e1a"),
            dict(range=[66, 100], color="#1a2e1a"),
        ],
    ),
), row=1, col=1)

fig_dash.add_trace(go.Indicator(
    mode="gauge+number",
    value=m_plan if m_plan else 60,
    number=dict(suffix=" meses", font=dict(size=36, color="white")),
    gauge=dict(
        axis=dict(range=[0, 48], tickwidth=1, tickcolor="#555", dtick=12),
        bar=dict(color="#3498db", thickness=0.75),
        bgcolor="#1a1a2e",
        borderwidth=1, bordercolor="#333",
        steps=[
            dict(range=[0, 12], color="#0a3d0a"),
            dict(range=[12, 24], color="#1a2e1a"),
            dict(range=[24, 36], color="#2e2e1a"),
            dict(range=[36, 48], color="#2e1a1a"),
        ],
        threshold=dict(line=dict(color="white", width=2), thickness=0.8, value=36),
    ),
), row=1, col=2)

fig_dash.update_layout(
    paper_bgcolor="#0d0d1a", font=dict(color="white"),
    height=280, margin=dict(t=50, b=20, l=40, r=40),
    showlegend=False,
)
for ann in fig_dash['layout']['annotations']:
    ann['font'] = dict(color='white', size=13)
fig_dash.show()

# ── Resumen en texto, directo ──
print("═" * 60)
print(f"  💰 Tenés hoy        : ${pat_actual:,.0f} USD")
print(f"  🎯 Querés llegar a  : ${META_USD:,} USD")
print(f"  📏 Te faltan        : ${brecha:,.0f} USD ({100-pct_avance:.0f}%)")
print(f"  💵 Aportás por mes  : ${APORTE_MENSUAL_USD}/mes")
if m_plan:
    años = m_plan / 12
    print(f"  🕐 Llegás en        : ~{m_plan} meses ({años:.1f} años)")
else:
    print(f"  🕐 Llegás en        : más de 5 años con este plan")
print("═" * 60)


## 📦 ¿Qué comprar en Bull Market?
> Tu plan concreto. Cada instrumento dice qué es, cuánto poner, y por qué.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  TU PORTAFOLIO CONCRETO — Solo cosas que comprás en Bull Market
# ═══════════════════════════════════════════════════════════════

# Pesos del portafolio Max Sharpe, mapeados a instrumentos BM
plan_bm = [
    {
        "instrumento": "ONs Corporativas USD",
        "que_es": "Le prestás plata a empresas grandes (Pampa, YPF, Telecom). Te devuelven con interés.",
        "peso": w_sharpe.get("ON_USD", 0.30),
        "donde": "Bull Market → Renta Fija → ONs en USD",
        "riesgo": "Bajo",
        "color": "#2ecc71",
    },
    {
        "instrumento": "FCI Corporativo HD",
        "que_es": "Un fondo que compra muchas ONs por vos. Más diversificado, menos trabajo.",
        "peso": w_sharpe.get("FCI_Corp_HD", 0.25),
        "donde": "Bull Market → Fondos → BM Ahorro Dólares o Allaria Dólar Dinámico",
        "riesgo": "Bajo",
        "color": "#f39c12",
    },
    {
        "instrumento": "SPY CEDEAR",
        "que_es": "Comprás un pedacito de las 500 empresas más grandes de USA (Apple, Google, Amazon...).",
        "peso": w_sharpe.get("SPY_CEDEAR", 0.15),
        "donde": "Bull Market → CEDEARs → SPY",
        "riesgo": "Medio-Alto",
        "color": "#3498db",
    },
    {
        "instrumento": "QQQ CEDEAR",
        "que_es": "Similar a SPY pero enfocado en tecnología (Apple, Microsoft, Nvidia, Meta).",
        "peso": w_sharpe.get("QQQ_CEDEAR", 0.10),
        "donde": "Bull Market → CEDEARs → QQQ",
        "riesgo": "Alto",
        "color": "#9b59b6",
    },
]

# Recalcular pesos para que sumen 100% (sacamos staking que era Binance)
total_peso = sum(p["peso"] for p in plan_bm)
for p in plan_bm:
    p["peso_norm"] = p["peso"] / total_peso
    p["monto"] = CAPITAL_LIQUIDO_USD * p["peso_norm"]
    p["aporte_mes"] = APORTE_MENSUAL_USD * p["peso_norm"]

# ── Gráfico: Donut + Tabla de montos ──
fig_plan = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=["Distribución del portafolio", "¿Cuánto poner en cada uno?"],
)

fig_plan.add_trace(go.Pie(
    labels=[p["instrumento"] for p in plan_bm],
    values=[p["peso_norm"] for p in plan_bm],
    marker=dict(colors=[p["color"] for p in plan_bm]),
    textinfo="label+percent",
    textfont=dict(size=11, color="white"),
    hole=0.45,
    hovertemplate="<b>%{label}</b><br>%{percent}<br>$%{value:,.0f}<extra></extra>",
), row=1, col=1)

fig_plan.add_trace(go.Bar(
    x=[p["instrumento"][:15] for p in plan_bm],
    y=[p["monto"] for p in plan_bm],
    marker_color=[p["color"] for p in plan_bm],
    text=[f"${p['monto']:,.0f}" for p in plan_bm],
    textposition="auto",
    textfont=dict(color="white", size=13),
    showlegend=False,
), row=1, col=2)

fig_plan.update_layout(
    title=dict(text=f"💰 Tu portafolio: ${CAPITAL_LIQUIDO_USD:,} USD distribuidos en Bull Market",
               font=dict(size=13, color="white")),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"), height=420,
    margin=dict(t=60, b=60, l=50, r=30), showlegend=False,
)
fig_plan.update_xaxes(color="white", tickangle=-15, row=1, col=2)
fig_plan.update_yaxes(color="white", tickformat="$,.0f", gridcolor="#333", row=1, col=2)
for ann in fig_plan['layout']['annotations']:
    ann['font'] = dict(color='white', size=12)
fig_plan.show()

# ── Detalle en texto ──
print("\n" + "═" * 68)
print("  📋 PLAN DETALLADO — Qué comprar en Bull Market")
print("═" * 68)
for p in plan_bm:
    print(f"\n  {'─'*64}")
    riesgo_emoji = {"Bajo": "🟢", "Medio-Alto": "🟡", "Alto": "🔴"}.get(p['riesgo'], "⚪")
    print(f"  {riesgo_emoji} {p['instrumento']}  ({p['peso_norm']:.0%} del portafolio)")
    print(f"  💬 {p['que_es']}")
    print(f"  💵 Invertir ahora   : ${p['monto']:>8,.0f}")
    print(f"  💵 Aportar por mes  : ${p['aporte_mes']:>8,.0f}")
    print(f"  📍 Dónde comprarlo  : {p['donde']}")
    print(f"  ⚡ Riesgo           : {p['riesgo']}")
print(f"\n  {'─'*64}")
print(f"  TOTAL INVERTIDO HOY  : ${CAPITAL_LIQUIDO_USD:>8,}")
print(f"  APORTE MENSUAL       : ${APORTE_MENSUAL_USD:>8}/mes")
print(f"  {'─'*64}")


## 🕐 ¿Cuándo llego a $3.000?
> Tu proyección con el plan de arriba. Muestra 3 escenarios:
> tu aporte actual ($55), si conseguís la pasantía ($100-150), y si no aportás nada.

In [ ]:
meses_eje = list(range(HORIZONTE_MESES + 1))

# Escenarios con el mismo portafolio (Max Sharpe) pero distintos aportes
escenarios = [
    (f"Hoy: ${APORTES_ESCENARIOS[0]}/mes",   APORTES_ESCENARIOS[0],  "#f39c12",  "solid"),
    (f"Pasantía: ${APORTES_ESCENARIOS[1]}/mes", APORTES_ESCENARIOS[1], "#00d4aa", "solid"),
    (f"Full: ${APORTES_ESCENARIOS[2]}/mes",   APORTES_ESCENARIOS[2],  "#3498db",  "solid"),
    ("Sin aportes",                            0,                      "#e74c3c",  "dot"),
]

fig_proy = go.Figure()

for nombre, aporte, color, dash in escenarios:
    serie = proyectar(CAPITAL_LIQUIDO_USD, ret_plan, HORIZONTE_MESES, aporte_mensual=aporte)
    m = mes_meta(serie, META_USD)
    tag = f" → ¡meta en {m} meses!" if m else ""
    fig_proy.add_trace(go.Scatter(
        x=meses_eje, y=serie, mode="lines",
        name=f"{nombre}{tag}",
        line=dict(color=color, width=2.5 if dash == "solid" else 2, dash=dash),
        hovertemplate=f"<b>{nombre}</b><br>Mes %{{x}}<br>${{y:,.0f}}<extra></extra>",
    ))
    if m and m <= HORIZONTE_MESES:
        fig_proy.add_trace(go.Scatter(
            x=[m], y=[META_USD], mode="markers+text",
            marker=dict(size=14, color=color, symbol="star", line=dict(color="white", width=2)),
            text=[f"  {m} meses"], textposition="middle right",
            textfont=dict(color=color, size=11),
            showlegend=False, hoverinfo="skip",
        ))

fig_proy.add_hline(y=META_USD, line_dash="dash", line_color="white", line_width=1.5,
    annotation_text=f"  Meta: ${META_USD:,}", annotation_position="top left",
    annotation_font=dict(color="white", size=12))
fig_proy.add_hline(y=pat_actual, line_dash="dot", line_color="#888", line_width=1,
    annotation_text=f"  Hoy: ${pat_actual:,}", annotation_position="bottom left",
    annotation_font=dict(color="#aaa", size=10))

fig_proy.update_layout(
    title=dict(text=f"🎯 ¿Cuándo llegás a ${META_USD:,}? — Depende de cuánto aportás por mes",
               font=dict(size=13, color="white")),
    xaxis=dict(title="Meses", color="white", gridcolor="#333", zeroline=False),
    yaxis=dict(title="Tu patrimonio (USD)", tickformat="$,.0f",
               color="white", gridcolor="#333", zeroline=False),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    legend=dict(bgcolor="#111", bordercolor="#555", borderwidth=1,
                x=0.01, y=0.99, xanchor="left", yanchor="top"),
    height=500, margin=dict(t=60, b=60, l=80, r=40),
)
fig_proy.show()

# ── Resumen en texto ──
print("\n" + "═" * 60)
print("  🕐 RESUMEN DE TIEMPOS")
print("═" * 60)
for nombre, aporte, color, dash in escenarios:
    serie = proyectar(CAPITAL_LIQUIDO_USD, ret_plan, 60, aporte_mensual=aporte)
    m = mes_meta(serie, META_USD)
    inv_total = CAPITAL_LIQUIDO_USD + aporte * (m if m else 36)
    if m:
        print(f"  ✅ {nombre:<25} → meta en {m:>2} meses ({m/12:.1f} años)")
    else:
        vf = serie[36]
        print(f"  ⏳ {nombre:<25} → ${vf:,.0f} en 3 años (no llega)")
print("═" * 60)


## 🚀 Plan de escalamiento: hoy $55 → pasantía $150
> Simulación: aportás $55/mes los primeros 6 meses, y después subís a $150/mes.

In [ ]:
# Simular escenario escalonado: 6 meses a $55, después $150
MESES_ANTES_PASANTIA = 6
APORTE_POST_PASANTIA = 150

def proyectar_escalonado(capital, ret_anual, meses_total,
                          aporte_fase1, meses_fase1, aporte_fase2,
                          otros=OTROS_ACTIVOS_USD):
    ret_mensual = (1 + ret_anual) ** (1/12) - 1
    serie = [capital + otros]
    cap = capital
    for m in range(1, meses_total + 1):
        aporte = aporte_fase1 if m <= meses_fase1 else aporte_fase2
        cap = cap * (1 + ret_mensual) + aporte
        serie.append(cap + otros)
    return serie

serie_escalon = proyectar_escalonado(
    CAPITAL_LIQUIDO_USD, ret_plan, HORIZONTE_MESES,
    APORTE_MENSUAL_USD, MESES_ANTES_PASANTIA, APORTE_POST_PASANTIA)
m_escalon = mes_meta(serie_escalon, META_USD)

serie_fijo55 = proyectar(CAPITAL_LIQUIDO_USD, ret_plan, HORIZONTE_MESES, aporte_mensual=APORTE_MENSUAL_USD)
serie_fijo150 = proyectar(CAPITAL_LIQUIDO_USD, ret_plan, HORIZONTE_MESES, aporte_mensual=150)

fig_esc = go.Figure()

fig_esc.add_trace(go.Scatter(
    x=meses_eje, y=serie_escalon, mode="lines",
    name=f"Escalonado: $55→$150 en mes {MESES_ANTES_PASANTIA}" + (f" → meta en mes {m_escalon}" if m_escalon else ""),
    line=dict(color="#00d4aa", width=3),
))
fig_esc.add_trace(go.Scatter(
    x=meses_eje, y=serie_fijo55, mode="lines",
    name=f"Siempre $55/mes",
    line=dict(color="#f39c12", width=2, dash="dash"),
))
fig_esc.add_trace(go.Scatter(
    x=meses_eje, y=serie_fijo150, mode="lines",
    name=f"Siempre $150/mes",
    line=dict(color="#3498db", width=2, dash="dash"),
))

# Marcar el punto de cambio
fig_esc.add_vline(x=MESES_ANTES_PASANTIA, line_dash="dot", line_color="#aaa", line_width=1,
    annotation_text=f"  Pasantía (mes {MESES_ANTES_PASANTIA})",
    annotation_position="top right",
    annotation_font=dict(color="#aaa", size=10))

if m_escalon:
    fig_esc.add_trace(go.Scatter(
        x=[m_escalon], y=[META_USD], mode="markers+text",
        marker=dict(size=14, color="#00d4aa", symbol="star", line=dict(color="white", width=2)),
        text=[f"  ¡Meta en mes {m_escalon}!"], textposition="middle right",
        textfont=dict(color="#00d4aa", size=12),
        showlegend=False,
    ))

fig_esc.add_hline(y=META_USD, line_dash="dash", line_color="white", line_width=1.5,
    annotation_text=f"  Meta: ${META_USD:,}", annotation_position="top left",
    annotation_font=dict(color="white", size=11))

fig_esc.update_layout(
    title=dict(text="🚀 Escenario realista: arrancás con $55 y subís a $150 cuando conseguís la pasantía",
               font=dict(size=13, color="white")),
    xaxis=dict(title="Meses", color="white", gridcolor="#333"),
    yaxis=dict(title="Patrimonio (USD)", tickformat="$,.0f", color="white", gridcolor="#333"),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    legend=dict(bgcolor="#111", bordercolor="#555", borderwidth=1,
                x=0.01, y=0.99),
    height=480, margin=dict(t=60, b=60, l=80, r=40),
)
fig_esc.show()

m_fijo55 = mes_meta(serie_fijo55, META_USD)
m_fijo150 = mes_meta(serie_fijo150, META_USD)
print("\n" + "═" * 60)
print("  🔑 IMPACTO DE LA PASANTÍA")
print("═" * 60)
if m_fijo55:
    print(f"  Siempre $55/mes  → meta en {m_fijo55} meses")
if m_escalon:
    print(f"  $55→$150 en mes 6 → meta en {m_escalon} meses")
    if m_fijo55:
        ahorro = m_fijo55 - m_escalon
        print(f"  🎯 ¡Te ahorrás {ahorro} meses consiguiendo la pasantía!")
if m_fijo150:
    print(f"  Siempre $150/mes → meta en {m_fijo150} meses (mejor caso)")
print("═" * 60)


## 📊 Detalle: Fondos Bull Market — Rendimientos reales (Abril 2026)
> Estos son los fondos corporativos en dólares que podés comprar en Bull Market.
> Los rendimientos son **reales** del informe de Research de Abril 2026.

In [ ]:
nombres_bm = list(FONDOS_CORP_HD.keys())
ret12 = [FONDOS_CORP_HD[n]["ret_12m"] for n in nombres_bm]
ret6  = [FONDOS_CORP_HD[n]["ret_6m"]  for n in nombres_bm]
ret3  = [FONDOS_CORP_HD[n]["ret_3m"]  for n in nombres_bm]

fig_bm = go.Figure()

fig_bm.add_trace(go.Bar(name="12 meses", x=nombres_bm, y=ret12,
    marker_color="#00d4aa", text=[f"{v:.1%}" for v in ret12], textposition="auto",
    textfont=dict(color="white", size=11)))
fig_bm.add_trace(go.Bar(name="6 meses",  x=nombres_bm, y=ret6,
    marker_color="#3498db", text=[f"{v:.1%}" for v in ret6],  textposition="auto",
    textfont=dict(color="white", size=11)))
fig_bm.add_trace(go.Bar(name="3 meses",  x=nombres_bm, y=ret3,
    marker_color="#9b59b6", text=[f"{v:.1%}" for v in ret3],  textposition="auto",
    textfont=dict(color="white", size=11)))

fig_bm.update_layout(
    barmode="group",
    title=dict(text="📊 Fondos Corporativos HD — Retornos en USD (Bull Market Research Abr 2026)",
               font=dict(size=13, color="white")),
    xaxis=dict(color="white", tickangle=-15),
    yaxis=dict(title="Retorno (%)", tickformat=".1%", color="white", gridcolor="#333"),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    legend=dict(bgcolor="#111", bordercolor="#555", borderwidth=1),
    height=450, margin=dict(t=60, b=60, l=80, r=40),
)
fig_bm.show()

print("\n📌 Los fondos corporativos HD invierten principalmente en ONs de:")
print("   Pampa Energía, John Deere, Loma Negra, YPF, Telecom, Vista Oil")
print("   Disponibles a través de Bull Market ALyC")

---
# 🔬 Detalle Técnico (opcional)
> Acá abajo están los gráficos técnicos del modelo: frontera de Markowitz,
> scatter riesgo/retorno, análisis de sensibilidad, etc.
> **No necesitás entenderlos para seguir el plan de arriba.**
> Están por si querés profundizar o mostrarle a alguien que sepa más.

## 🎨 7. Visualizaciones
### 7A. Tabla Comparativa Rankeada

In [ ]:
def color_neto(v):
    if v >= 0.06:  return "#27ae60"
    elif v >= 0.03: return "#f39c12"
    elif v >= 0.0:  return "#e67e22"
    else:           return "#c0392b"

fill_neto = [color_neto(v) for v in df_assets["retorno_neto"]]

fig_tabla = go.Figure(data=[go.Table(
    columnwidth=[160, 120, 130, 110, 100, 100],
    header=dict(
        values=["<b>Instrumento</b>", "<b>Retorno Bruto</b>", "<b>Retorno Neto Real</b>",
                "<b>Volatilidad</b>", "<b>Risk Score</b>", "<b>Sharpe</b>"],
        fill_color="#1a1a2e",
        font=dict(color="white", size=12, family="Arial"),
        align="center", height=35,
        line_color="#333",
    ),
    cells=dict(
        values=[
            df_assets["activo"],
            [f"{v:.1%}" for v in df_assets["retorno_bruto"]],
            [f"{v:.1%}" for v in df_assets["retorno_neto"]],
            [f"{v:.1%}" for v in df_assets["volatilidad"]],
            [f"{v:.2f}" for v in df_assets["risk_score"]],
            [f"{v:.2f}" for v in df_assets["sharpe"]],
        ],
        fill_color=[
            ["#16213e"] * len(df_assets),
            ["#16213e"] * len(df_assets),
            fill_neto,
            ["#16213e"] * len(df_assets),
            ["#16213e"] * len(df_assets),
            ["#16213e"] * len(df_assets),
        ],
        font=dict(color="white", size=11, family="Arial"),
        align="center", height=30,
        line_color="#222",
    )
)])

fig_tabla.update_layout(
    title=dict(text="📊 Comparativa de Instrumentos — Ordenado por Retorno Neto Real",
               font=dict(size=14, color="white")),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    margin=dict(t=50, b=10, l=10, r=10),
    height=280,
)
fig_tabla.show()

### 7B. Scatter — Riesgo vs. Retorno Neto Real

In [ ]:
peso_lookup = {
    "SPY_CEDEAR":   w_sharpe.get("SPY_CEDEAR", 0.15),
    "QQQ_CEDEAR":   w_sharpe.get("QQQ_CEDEAR", 0.15),
    "ON_USD":       w_sharpe.get("ON_USD",     0.25),
    "FCI_Corp_HD":  w_sharpe.get("FCI_Corp_HD", 0.25),
    "Staking_CeFi": w_sharpe.get("Staking_CeFi", 0.20),
}
activo_key_map = {
    "ON USD": "ON_USD", "SPY CEDEAR": "SPY_CEDEAR", "QQQ CEDEAR": "QQQ_CEDEAR",
    "FCI Corp HD": "FCI_Corp_HD", "Staking CeFi": "Staking_CeFi",
}
colores = {
    "ON USD": "#2ecc71", "SPY CEDEAR": "#3498db", "QQQ CEDEAR": "#9b59b6",
    "FCI Corp HD": "#f39c12", "Staking CeFi": "#e74c3c",
}

fig_scatter = go.Figure()

for _, row in df_assets.iterrows():
    key  = activo_key_map.get(row["activo"], "")
    peso = peso_lookup.get(key, 0.20)
    capital_asignado = CAPITAL_LIQUIDO_USD * peso

    fig_scatter.add_trace(go.Scatter(
        x=[row["risk_score"]],
        y=[row["retorno_neto"]],
        mode="markers+text",
        marker=dict(
            size=max(peso * 300, 30),
            color=colores.get(row["activo"], "white"),
            opacity=0.85,
            line=dict(color="white", width=1.5),
        ),
        text=[f"<b>{row['activo']}</b><br>${capital_asignado:,.0f}"],
        textposition="top center",
        textfont=dict(size=10, color="white"),
        name=row["activo"],
        hovertemplate=(
            f"<b>{row['activo']}</b><br>"
            f"Risk Score: {row['risk_score']:.2f}<br>"
            f"Retorno neto: {row['retorno_neto']:.2%}<br>"
            f"Peso portafolio: {peso:.1%}<br>"
            f"Capital: ${capital_asignado:,.0f}<extra></extra>"
        ),
    ))

mid_risk = df_assets["risk_score"].mean()
fig_scatter.add_vline(x=mid_risk, line_dash="dot", line_color="#555", line_width=1)
fig_scatter.add_hline(y=0.0, line_dash="dash", line_color="#e74c3c", line_width=1.5,
                      annotation_text="Umbral real (0%)", annotation_position="bottom right",
                      annotation_font_color="#e74c3c")

fig_scatter.update_layout(
    title=dict(text="⚖️ Riesgo vs. Retorno Neto Real — Burbuja = peso en portafolio óptimo",
               font=dict(size=13, color="white")),
    xaxis=dict(title="Risk Score (menor = más seguro)", color="white", gridcolor="#333", zeroline=False),
    yaxis=dict(title="Retorno Neto Real Anual", tickformat=".1%", color="white", gridcolor="#333", zeroline=False),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"), showlegend=False,
    height=520, margin=dict(t=60, b=60, l=80, r=40),
)
fig_scatter.show()

### 7C. Frontera Eficiente de Markowitz

In [ ]:
targets  = np.linspace(mu_pf.min() * 1.01, mu_pf.max() * 0.92, 400)
f_vols, f_rets = [], []

for t in targets:
    try:
        ef_t = EfficientFrontier(mu_pf, S_pf)
        ef_t.add_constraint(lambda w: w >= 0.0)
        ef_t.add_constraint(lambda w: w <= 0.75)
        ef_t.efficient_return(target_return=float(t))
        r, v, _ = ef_t.portfolio_performance(risk_free_rate=RISK_FREE_RATE)
        f_rets.append(r); f_vols.append(v)
    except:
        pass

fig_ef = go.Figure()

fig_ef.add_trace(go.Scatter(
    x=f_vols, y=f_rets, mode="lines",
    name="Frontera Eficiente",
    line=dict(color="#00d4aa", width=3),
))

for _, row in df_assets.iterrows():
    fig_ef.add_trace(go.Scatter(
        x=[row["volatilidad"]], y=[row["retorno_neto"]],
        mode="markers+text",
        marker=dict(size=10, color=colores.get(row["activo"], "white"),
                    line=dict(color="white", width=1)),
        text=[row["activo"]],
        textposition="top center",
        textfont=dict(size=9, color="white"),
        name=row["activo"], showlegend=False,
        hovertemplate=f"<b>{row['activo']}</b><br>Vol: {row['volatilidad']:.2%}<br>Ret neto: {row['retorno_neto']:.2%}<extra></extra>",
    ))

fig_ef.add_trace(go.Scatter(
    x=[vol_ms], y=[ret_ms], mode="markers+text",
    marker=dict(size=18, color="gold", symbol="star", line=dict(color="white", width=1.5)),
    text=["Max Sharpe"], textposition="top right",
    textfont=dict(size=10, color="gold"),
    name=f"Máx. Sharpe  ({ret_ms:.1%} ret / {vol_ms:.1%} vol)",
))

fig_ef.add_trace(go.Scatter(
    x=[vol_mv], y=[ret_mv], mode="markers+text",
    marker=dict(size=18, color="#ff6b6b", symbol="diamond", line=dict(color="white", width=1.5)),
    text=["Mín. Vol."], textposition="top right",
    textfont=dict(size=10, color="#ff6b6b"),
    name=f"Mín. Volatilidad  ({ret_mv:.1%} ret / {vol_mv:.1%} vol)",
))

fig_ef.add_trace(go.Scatter(
    x=[0, vol_ms * 1.5], y=[RISK_FREE_RATE, RISK_FREE_RATE + sharpe_ms * vol_ms * 1.5],
    mode="lines", line=dict(color="gold", width=1, dash="dot"),
    name="Capital Market Line", opacity=0.6,
))

fig_ef.update_layout(
    title=dict(text="📈 Frontera Eficiente de Markowitz — Portafolio óptimo",
               font=dict(size=13, color="white")),
    xaxis=dict(title="Volatilidad Anual", tickformat=".0%", color="white", gridcolor="#333", zeroline=False),
    yaxis=dict(title="Retorno Esperado", tickformat=".0%", color="white", gridcolor="#333", zeroline=False),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    legend=dict(bgcolor="#111", bordercolor="#555", borderwidth=1),
    height=580, margin=dict(t=60, b=60, l=80, r=40),
)
fig_ef.show()

### 7D. Proyección de Crecimiento hacia la Meta
> **🆕 v2:** Compara los 3 escenarios de aporte mensual ($40, $70, $150 USD)
> sobre el portafolio Max Sharpe, contra el status quo en Binance.

In [ ]:
def proyectar(capital_liq, ret_anual, meses, aporte_mensual=0,
              otros=OTROS_ACTIVOS_USD):
    """Proyecta patrimonio total mes a mes con capitalización compuesta + aportes."""
    ret_mensual = (1 + ret_anual) ** (1/12) - 1
    serie = [capital_liq + otros]
    capital = capital_liq
    for m in range(1, meses + 1):
        capital = capital * (1 + ret_mensual) + aporte_mensual
        serie.append(capital + otros)
    return serie

meses_eje = list(range(HORIZONTE_MESES + 1))

ret_ms_neto = ret_ms - INFLACION_USD_ANUAL
ret_mv_neto = ret_mv - INFLACION_USD_ANUAL
ret_binance = APR_STAKING_BINANCE - INFLACION_USD_ANUAL

def mes_meta(serie, meta):
    for i, v in enumerate(serie):
        if v >= meta: return i
    return None

# ═══════════════════════════════════════════════════════════════════
#  GRÁFICO PRINCIPAL: Max Sharpe × 3 escenarios de aporte mensual
#  + Status quo Binance (sin aportes) como línea base
# ═══════════════════════════════════════════════════════════════════
colores_aporte = {40: "#00d4aa", 70: "#3498db", 150: "#9b59b6"}
dash_aporte    = {40: "solid",   70: "solid",   150: "solid"}

fig_proy = go.Figure()

# ── Escenarios Max Sharpe con cada aporte ──
for aporte in APORTES_ESCENARIOS:
    serie = proyectar(CAPITAL_LIQUIDO_USD, ret_ms_neto, HORIZONTE_MESES, aporte_mensual=aporte)
    m = mes_meta(serie, META_USD)
    tag = f" → meta en mes {m} ({m/12:.1f} a)" if m else " → no alcanza"
    color = colores_aporte.get(aporte, "#00d4aa")
    fig_proy.add_trace(go.Scatter(
        x=meses_eje, y=serie, mode="lines",
        name=f"Max Sharpe + ${aporte}/mes{tag}",
        line=dict(color=color, width=2.5),
        hovertemplate=f"<b>Max Sharpe + ${aporte}/mes</b><br>Mes %{{x}}<br>${{y:,.0f}}<extra></extra>",
    ))
    if m and m <= HORIZONTE_MESES:
        fig_proy.add_trace(go.Scatter(
            x=[m], y=[META_USD], mode="markers",
            marker=dict(size=12, color=color, symbol="circle", line=dict(color="white", width=2)),
            showlegend=False, hoverinfo="skip",
        ))

# ── Mín. Volatilidad con aporte base ──
serie_mv = proyectar(CAPITAL_LIQUIDO_USD, ret_mv_neto, HORIZONTE_MESES, aporte_mensual=APORTE_MENSUAL_USD)
m_mv = mes_meta(serie_mv, META_USD)
tag_mv = f" → meta en mes {m_mv} ({m_mv/12:.1f} a)" if m_mv else " → no alcanza"
fig_proy.add_trace(go.Scatter(
    x=meses_eje, y=serie_mv, mode="lines",
    name=f"Mín. Vol. + ${APORTE_MENSUAL_USD}/mes{tag_mv}",
    line=dict(color="#ffd700", width=2, dash="dash"),
    hovertemplate=f"<b>Mín. Vol. + ${APORTE_MENSUAL_USD}/mes</b><br>Mes %{{x}}<br>${{y:,.0f}}<extra></extra>",
))

# ── Status quo Binance SIN aportes ──
serie_bce = proyectar(CAPITAL_LIQUIDO_USD, ret_binance, HORIZONTE_MESES, aporte_mensual=0)
fig_proy.add_trace(go.Scatter(
    x=meses_eje, y=serie_bce, mode="lines",
    name=f"Binance sin aportes (status quo)",
    line=dict(color="#e74c3c", width=2, dash="dot"),
    hovertemplate=f"<b>Binance (status quo)</b><br>Mes %{{x}}<br>${{y:,.0f}}<extra></extra>",
))

# ── Líneas de referencia ──
fig_proy.add_hline(y=META_USD, line_dash="dash", line_color="white", line_width=1.5,
    annotation_text=f"  Meta: ${META_USD:,}", annotation_position="top left",
    annotation_font=dict(color="white", size=11))
fig_proy.add_hline(y=CAPITAL_LIQUIDO_USD + OTROS_ACTIVOS_USD,
    line_dash="dot", line_color="#888", line_width=1,
    annotation_text=f"  Actual: ${CAPITAL_LIQUIDO_USD + OTROS_ACTIVOS_USD:,}",
    annotation_position="bottom left", annotation_font=dict(color="#aaa", size=10))

fig_proy.update_layout(
    title=dict(text=f"🎯 Proyección hacia ${META_USD:,} USD — Max Sharpe × 3 escenarios de aporte",
               font=dict(size=13, color="white")),
    xaxis=dict(title="Meses", color="white", gridcolor="#333", zeroline=False),
    yaxis=dict(title="Patrimonio Total (USD)", tickformat="$,.0f",
               color="white", gridcolor="#333", zeroline=False),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    legend=dict(bgcolor="#111", bordercolor="#555", borderwidth=1,
                x=0.01, y=0.99, xanchor="left", yanchor="top"),
    height=560, margin=dict(t=60, b=60, l=80, r=40),
)
fig_proy.show()

# ─── Resumen Final ──────────────────────────────────────────────────────
print("\n" + "═"*72)
print("  📋 RESUMEN EJECUTIVO — ESCENARIOS DE APORTE MENSUAL")
print("═"*72)
print(f"  Patrimonio actual: ${CAPITAL_LIQUIDO_USD + OTROS_ACTIVOS_USD:,.0f} USD  |  Meta: ${META_USD:,} USD")
print(f"  Retorno neto Max Sharpe: {ret_ms_neto:+.2%}  |  Retorno neto Mín. Vol.: {ret_mv_neto:+.2%}")
print()
print(f"  {'Escenario':<40} {'Aporte':>8} {'Mes meta':>10} {'Años':>8} {'Valor final':>14}")
print("  " + "-"*82)
for aporte in APORTES_ESCENARIOS:
    serie = proyectar(CAPITAL_LIQUIDO_USD, ret_ms_neto, HORIZONTE_MESES, aporte_mensual=aporte)
    m = mes_meta(serie, META_USD)
    val_final = serie[-1]
    inv_total = CAPITAL_LIQUIDO_USD + aporte * HORIZONTE_MESES
    if m:
        print(f"  ✅ Max Sharpe + ${aporte}/mes{'':<20} ${aporte:>6}/mes   mes {m:>3}   {m/12:>5.1f}a   ${val_final:>11,.0f}")
    else:
        print(f"  ⏳ Max Sharpe + ${aporte}/mes{'':<20} ${aporte:>6}/mes   {'—':>5}   {'—':>5}   ${val_final:>11,.0f}")

# Mín Vol con aporte base
m_mv2 = mes_meta(serie_mv, META_USD)
vf_mv = serie_mv[-1]
if m_mv2:
    print(f"  ✅ Mín. Vol. + ${APORTE_MENSUAL_USD}/mes{'':<21} ${APORTE_MENSUAL_USD:>6}/mes   mes {m_mv2:>3}   {m_mv2/12:>5.1f}a   ${vf_mv:>11,.0f}")
else:
    print(f"  ⏳ Mín. Vol. + ${APORTE_MENSUAL_USD}/mes{'':<21} ${APORTE_MENSUAL_USD:>6}/mes   {'—':>5}   {'—':>5}   ${vf_mv:>11,.0f}")

# Binance sin aportes
vf_bce = serie_bce[-1]
print(f"  ❌ Binance (status quo){'':<24} ${'0':>5}/mes   {'—':>5}   {'—':>5}   ${vf_bce:>11,.0f}")
print("═"*72)
print()
print(f"  💡 Inversión total en {HORIZONTE_MESES} meses con cada aporte:")
for aporte in APORTES_ESCENARIOS:
    inv_total = CAPITAL_LIQUIDO_USD + aporte * HORIZONTE_MESES
    print(f"     ${aporte}/mes → ${inv_total:,.0f} invertidos  (capital + aportes)")


### 7E. 🆕 Análisis de Sensibilidad — APR vs. Meses hasta la Meta

> Muestra cómo varía la fecha de llegada a la meta según diferentes APRs,
> para cada nivel de aporte mensual ($40, $70, $150).

In [ ]:
apr_range = np.arange(0.01, 0.16, 0.005)

fig_sens = go.Figure()
colores_sens = {0: "#e74c3c", 40: "#f39c12", 70: "#00d4aa", 150: "#3498db"}

for aporte in [0] + APORTES_ESCENARIOS:
    meses_meta = []
    for apr in apr_range:
        ret_real = apr - INFLACION_USD_ANUAL
        s = proyectar(CAPITAL_LIQUIDO_USD, ret_real, 60, aporte_mensual=aporte)
        m = mes_meta(s, META_USD)
        meses_meta.append(m if m else 60)

    label = f"${aporte}/mes" if aporte > 0 else "Sin aportes"
    fig_sens.add_trace(go.Scatter(
        x=apr_range * 100,  # eje numérico en %
        y=meses_meta,
        mode="lines+markers",
        name=label,
        line=dict(color=colores_sens.get(aporte, "#aaa"), width=2.5,
                  dash="dot" if aporte == 0 else "solid"),
        marker=dict(size=4),
        hovertemplate=f"{label}<br>APR: %{{x:.1f}}%<br>Meses: %{{y}}<extra></extra>",
    ))

# Marcar APRs clave con vlines numéricas
marcas = [
    (APR_STAKING_BINANCE * 100, "Binance", "#e74c3c"),
    (fci_hd_bruto * 100, "FCI Corp HD", "#f39c12"),
    (best_on_bruto * 100, "Mejor ON", "#2ecc71"),
    (ret_ms * 100, "Max Sharpe", "#00d4aa"),
]
for apr_val, label, color in marcas:
    fig_sens.add_vline(x=apr_val, line_dash="dot", line_color=color, line_width=1,
                       annotation_text=f"  {label}", annotation_position="top left",
                       annotation_font=dict(color=color, size=9))

fig_sens.add_hline(y=HORIZONTE_MESES, line_dash="dash", line_color="white", line_width=1,
                   annotation_text=f"Horizonte: {HORIZONTE_MESES}m",
                   annotation_position="bottom right",
                   annotation_font=dict(color="#aaa", size=9))

fig_sens.update_layout(
    title=dict(text="🔬 Sensibilidad: meses hasta $3.000 según APR y aporte mensual",
               font=dict(size=13, color="white")),
    xaxis=dict(title="APR Bruto del Portafolio (%)", color="white", gridcolor="#333",
               ticksuffix="%", dtick=2),
    yaxis=dict(title="Meses hasta la meta", color="white", gridcolor="#333",
               zeroline=False, range=[0, 62]),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    legend=dict(bgcolor="#111", bordercolor="#555", borderwidth=1),
    height=520, margin=dict(t=60, b=80, l=80, r=40),
)
fig_sens.show()

# ── Tabla resumen de sensibilidad ──
print("\n" + "═"*72)
print("  📊 TABLA DE SENSIBILIDAD — Meses hasta $3.000 USD")
print("═"*72)
print(f"  {'APR →':<18}", end="")
aprs_clave = [APR_STAKING_BINANCE, 0.05, ret_ms, 0.10, 0.12]
for apr in aprs_clave:
    print(f"{apr:>9.1%}", end="")
print()
print("  " + "-"*63)
for aporte in [0] + APORTES_ESCENARIOS:
    label = f"${aporte}/mes" if aporte > 0 else "Sin aportes"
    print(f"  {label:<18}", end="")
    for apr in aprs_clave:
        ret_real = apr - INFLACION_USD_ANUAL
        s = proyectar(CAPITAL_LIQUIDO_USD, ret_real, 60, aporte_mensual=aporte)
        m = mes_meta(s, META_USD)
        print(f"{'mes '+str(m) if m else '>60':>9}", end="")
    print()
print("═"*72)


### 7F. 🆕 Radar de Riesgo por Instrumento

In [ ]:
fig_radar = go.Figure()

colores_radar = {
    "ON USD": "#2ecc71", "SPY CEDEAR": "#3498db", "QQQ CEDEAR": "#9b59b6",
    "FCI Corp HD": "#f39c12", "Staking CeFi": "#e74c3c",
}

for _, row in df_risk.iterrows():
    activo = row["activo"]
    vals = [row[d] for d in dims] + [row[dims[0]]]  # cerrar el polígono
    cats = [d.capitalize() for d in dims] + [dims[0].capitalize()]

    fig_radar.add_trace(go.Scatterpolar(
        r=vals,
        theta=cats,
        fill="toself",
        name=activo,
        line=dict(color=colores_radar.get(activo, "white"), width=2),
        opacity=0.65,
    ))

fig_radar.update_layout(
    polar=dict(
        radialaxis=dict(visible=True, range=[0, 10], tickfont=dict(color="#888", size=9),
                        gridcolor="#333"),
        angularaxis=dict(tickfont=dict(color="white", size=11), gridcolor="#444"),
        bgcolor="#0d0d1a",
    ),
    title=dict(text="🛡️ Radar de Riesgo — 5 Dimensiones por Instrumento",
               font=dict(size=13, color="white")),
    paper_bgcolor="#0d0d1a",
    font=dict(color="white"),
    legend=dict(bgcolor="#111", bordercolor="#555", borderwidth=1),
    height=520, margin=dict(t=60, b=40, l=80, r=80),
)
fig_radar.show()

### 7G. 🆕 Asignación de Capital — Portafolio Max Sharpe
> Muestra en pesos concretos cuánto invertir en cada instrumento.

In [ ]:
labels_alloc = list(w_sharpe.keys())
values_alloc = [w_sharpe[k] for k in labels_alloc]
capital_alloc = [CAPITAL_LIQUIDO_USD * v for v in values_alloc]
colores_alloc = ["#2ecc71", "#3498db", "#9b59b6", "#f39c12", "#e74c3c"]

fig_alloc = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "pie"}, {"type": "bar"}]],
    subplot_titles=["Distribución Porcentual", "Capital Asignado (USD)"],
)

fig_alloc.add_trace(go.Pie(
    labels=labels_alloc,
    values=values_alloc,
    marker=dict(colors=colores_alloc[:len(labels_alloc)]),
    textinfo="label+percent",
    textfont=dict(size=11, color="white"),
    hole=0.4,
), row=1, col=1)

fig_alloc.add_trace(go.Bar(
    x=labels_alloc,
    y=capital_alloc,
    marker_color=colores_alloc[:len(labels_alloc)],
    text=[f"${v:,.0f}" for v in capital_alloc],
    textposition="auto",
    textfont=dict(color="white", size=12),
    showlegend=False,
), row=1, col=2)

fig_alloc.update_layout(
    title=dict(text=f"💰 Asignación Portafolio Max Sharpe — ${CAPITAL_LIQUIDO_USD:,} USD",
               font=dict(size=13, color="white")),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    height=450, margin=dict(t=60, b=40, l=60, r=40),
    showlegend=False,
)
fig_alloc.update_xaxes(color="white", gridcolor="#333", row=1, col=2)
fig_alloc.update_yaxes(color="white", gridcolor="#333", tickformat="$,.0f", row=1, col=2)

for ann in fig_alloc['layout']['annotations']:
    ann['font'] = dict(color='white', size=12)

fig_alloc.show()

### 7H. 🆕 Detalle Fondos Bull Market — Corporativos HD (Abril 2026)
> Retornos reales por período del Informe de Research.

In [ ]:
nombres_bm = list(FONDOS_CORP_HD.keys())
ret12 = [FONDOS_CORP_HD[n]["ret_12m"] for n in nombres_bm]
ret6  = [FONDOS_CORP_HD[n]["ret_6m"]  for n in nombres_bm]
ret3  = [FONDOS_CORP_HD[n]["ret_3m"]  for n in nombres_bm]

fig_bm = go.Figure()

fig_bm.add_trace(go.Bar(name="12 meses", x=nombres_bm, y=ret12,
    marker_color="#00d4aa", text=[f"{v:.1%}" for v in ret12], textposition="auto",
    textfont=dict(color="white", size=11)))
fig_bm.add_trace(go.Bar(name="6 meses",  x=nombres_bm, y=ret6,
    marker_color="#3498db", text=[f"{v:.1%}" for v in ret6],  textposition="auto",
    textfont=dict(color="white", size=11)))
fig_bm.add_trace(go.Bar(name="3 meses",  x=nombres_bm, y=ret3,
    marker_color="#9b59b6", text=[f"{v:.1%}" for v in ret3],  textposition="auto",
    textfont=dict(color="white", size=11)))

fig_bm.update_layout(
    barmode="group",
    title=dict(text="📊 Fondos Corporativos HD — Retornos en USD (Bull Market Research Abr 2026)",
               font=dict(size=13, color="white")),
    xaxis=dict(color="white", tickangle=-15),
    yaxis=dict(title="Retorno (%)", tickformat=".1%", color="white", gridcolor="#333"),
    paper_bgcolor="#0d0d1a", plot_bgcolor="#0d0d1a",
    font=dict(color="white"),
    legend=dict(bgcolor="#111", bordercolor="#555", borderwidth=1),
    height=450, margin=dict(t=60, b=60, l=80, r=40),
)
fig_bm.show()

print("\n📌 Los fondos corporativos HD invierten principalmente en ONs de:")
print("   Pampa Energía, John Deere, Loma Negra, YPF, Telecom, Vista Oil")
print("   Disponibles a través de Bull Market ALyC")

---
## ⚠️ Disclaimer

- Las proyecciones están basadas en **retornos históricos** y datos del informe de Bull Market Research (Abril 2026).
- **No constituyen asesoramiento financiero** ni recomendación de compra/venta.
- Rendimientos pasados no garantizan rendimientos futuros.
- Consultá con un asesor financiero matriculado antes de tomar decisiones de inversión.

---
*portfolio_optimizer v2.1 · Personalizado para Fran · Bull Market ALyC · Abril 2026*